In [26]:
import pandas as pd
import numpy as np

In [27]:
df = pd.read_csv("anime-dataset-2023.csv")

In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24905 entries, 0 to 24904
Data columns (total 24 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   anime_id      24905 non-null  int64
 1   Name          24905 non-null  str  
 2   English name  24905 non-null  str  
 3   Other name    24905 non-null  str  
 4   Score         24905 non-null  str  
 5   Genres        24905 non-null  str  
 6   Synopsis      24905 non-null  str  
 7   Type          24905 non-null  str  
 8   Episodes      24905 non-null  str  
 9   Aired         24905 non-null  str  
 10  Premiered     24905 non-null  str  
 11  Status        24905 non-null  str  
 12  Producers     24905 non-null  str  
 13  Licensors     24905 non-null  str  
 14  Studios       24905 non-null  str  
 15  Source        24905 non-null  str  
 16  Duration      24905 non-null  str  
 17  Rating        24905 non-null  str  
 18  Rank          24905 non-null  str  
 19  Popularity    24905 non-null  int64


In [29]:
df.columns

Index(['anime_id', 'Name', 'English name', 'Other name', 'Score', 'Genres',
       'Synopsis', 'Type', 'Episodes', 'Aired', 'Premiered', 'Status',
       'Producers', 'Licensors', 'Studios', 'Source', 'Duration', 'Rating',
       'Rank', 'Popularity', 'Favorites', 'Scored By', 'Members', 'Image URL'],
      dtype='str')

In [30]:
df["Score"] = pd.to_numeric(df["Score"], errors="coerce")
df["Scored By"] = pd.to_numeric(df["Scored By"], errors="coerce")

In [31]:
print(df["Score"].isna().sum())
print(df["Scored By"].isna().sum())

9213
9213


In [32]:
df["Score_missing"] = df["Score"].isna().astype(int)

df["Score"] = df["Score"].fillna(df["Score"].median())
df["Scored By"] = df["Scored By"].fillna(0)

In [33]:
df["Scored By"].describe()

count    2.490500e+04
mean     1.888608e+04
std      9.393985e+04
min      0.000000e+00
25%      0.000000e+00
50%      2.920000e+02
75%      3.345000e+03
max      2.660903e+06
Name: Scored By, dtype: float64

In [34]:
R = df["Score"]
v = df["Scored By"]
C = df["Score"].mean()
m = df["Scored By"].quantile(0.75)

In [35]:
df["weighted_score"] = (R*v + C*m)/(v+m)

print(df[["Name","weighted_score"]].head())

                              Name  weighted_score
0                     Cowboy Bebop        8.741375
1  Cowboy Bebop: Tengoku no Tobira        8.348149
2                           Trigun        8.202947
3               Witch Hunter Robin        7.187283
4                   Bouken Ou Beet        6.749495


In [36]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df["Popularity"] = df["Popularity"].max() - df["Popularity"]

cols = ["Popularity", "Members", "Favorites"]

df[cols] = scaler.fit_transform(df[cols])

In [37]:
df["weighted_score"] = scaler.fit_transform(df[["weighted_score"]])

df["final_score"] = (
    0.85 * df["weighted_score"] +
    0.15 * df["Popularity"]
)

In [38]:
revised_df = df[["Name","Score","Scored By","weighted_score","Popularity","Members","Favorites","final_score"]].copy()

final_df = revised_df.sort_values(by = "final_score", ascending= False)

final_df.head(10)

,Name,Score,Scored By,weighted_score,Popularity,Members,Favorites,final_score
3961,Fullmetal Alchemist: Brotherhood,9.10,2020030.0,1.000000,0.999879,0.848317,1.000000,0.999982
5667,Steins;Gate,9.07,1336233.0,0.995217,0.999474,0.651714,0.840804,0.995856
14865,Shingeki no Kyojin Season 3 Part 2,9.05,1471825.0,0.992346,0.999029,0.561889,0.253876,0.993349
6456,Hunter x Hunter (2011),9.04,1651790.0,0.990963,0.999596,0.709532,0.920310,0.992257
17572,Kaguya-sama wa Kokurasetai: Ultra Romantic,9.05,451187.0,0.990332,0.991991,0.219157,0.133811,0.990580
9880,Gintama°,9.06,237957.0,0.989222,0.986612,0.159103,0.073284,0.988830
16617,Bleach: Sennen Kessen-hen,9.07,213872.0,0.990073,0.981232,0.118893,0.082714,0.988747
5989,Gintama',9.04,226175.0,0.986014,0.984387,0.140388,0.035684,0.985770
22348,Shingeki no Kyojin: The Final Season - Kankets...,9.05,155773.0,0.984925,0.980625,0.116349,0.041718,0.984280
7240,Gintama': Enchousen,9.03,157644.0,0.982114,0.971080,0.082590,0.013635,0.980459


In [39]:
print(len(df[df["Genres"] == "UNKNOWN"]))

4929


In [40]:
genre_df = df[df["Genres"].str.upper() != "UNKNOWN"].copy()

In [41]:
unique_genres = (
    genre_df["Genres"]
    .dropna()
    .str.split(", ")
    .explode()
    .str.strip()
    .unique()
)

unique_genres = sorted(unique_genres)

print(unique_genres)
print("\nTotal genres:", len(unique_genres))

['Action', 'Adventure', 'Avant Garde', 'Award Winning', 'Boys Love', 'Comedy', 'Drama', 'Ecchi', 'Erotica', 'Fantasy', 'Girls Love', 'Gourmet', 'Hentai', 'Horror', 'Mystery', 'Romance', 'Sci-Fi', 'Slice of Life', 'Sports', 'Supernatural', 'Suspense']

Total genres: 21


In [42]:
from rapidfuzz import process, fuzz

def get_best_genre_match(genre, unique_genres, score_cutoff=70):
    genre = genre.lower().strip()

    # Create a mapping of lowercase genre -> original genre
    genre_map = {g.lower(): g for g in unique_genres}

    # Exact match
    if genre in genre_map:
        return genre_map[genre]

    # Fuzzy match
    match = process.extractOne(
        genre,
        genre_map.keys(),
        scorer=fuzz.WRatio,
        score_cutoff=score_cutoff
    )

    if match:
        matched_genre, score, _ = match
        print(f"Genre found. Using '{genre_map[matched_genre]}' ({score:.1f}% match)")
        return genre_map[matched_genre]

    return None

In [43]:
genre_df["Genres"] = (
    genre_df["Genres"]
    .str.lower()
    .str.split(",")
    .apply(lambda genres: [g.strip() for g in genres])
)

In [54]:
genre_df["Genre_Set"] = genre_df["Genres"].apply(set)

In [55]:
def recommend_by_genres(genres, top_n=10):

    if isinstance(genres, str):
        genres = [genres]

    # Fuzzy match each input genre
    matched_genres = []
    for genre in genres:
        match = get_best_genre_match(genre, unique_genres)
        if match:
            matched_genres.append(match)

    if not matched_genres:
        print("No valid genres found.")
        return None

    selected = {g.lower().strip() for g in matched_genres}

    recommendations = genre_df.copy(deep=False)

    # Genre Match Score
    recommendations["genre_score"] = recommendations["Genre_Set"].apply(
        lambda x: len(selected & x) / len(selected)
    )

    recommendations = recommendations[
        recommendations["genre_score"] > 0
    ]

    recommendations["recommendation_score"] = (
        0.7 * recommendations["genre_score"] +
        0.3 * recommendations["final_score"]
    )

    return (
        recommendations
        .nlargest(top_n, "recommendation_score")
        [["Name",
          "Genres",
          "genre_score",
          "final_score",
          "recommendation_score"]]
        .reset_index(drop=True)
    )

In [56]:
recommend_by_genres(["romance"], 20)

,Name,Genres,genre_score,final_score,recommendation_score
0,Kaguya-sama wa Kokurasetai: Ultra Romantic,"[comedy, romance]",1.0,0.990580,0.997174
1,Fruits Basket: The Final,"[drama, romance, supernatural]",1.0,0.979827,0.993948
2,Clannad: After Story,"[drama, romance, supernatural]",1.0,0.976752,0.993026
3,Monogatari Series: Second Season,"[comedy, mystery, romance, supernatural]",1.0,0.954267,0.986280
4,Kaguya-sama wa Kokurasetai: First Kiss wa Owar...,"[comedy, drama, romance]",1.0,0.951847,0.985554
5,Howl no Ugoku Shiro,"[adventure, award winning, drama, fantasy, rom...",1.0,0.943353,0.983006
6,Shigatsu wa Kimi no Uso,"[drama, romance]",1.0,0.942904,0.982871
7,Rurouni Kenshin: Meiji Kenkaku Romantan - Tsui...,"[action, drama, romance]",1.0,0.939154,0.981746
8,Seishun Buta Yarou wa Yumemiru Shoujo no Yume ...,"[drama, romance, supernatural]",1.0,0.933550,0.980065
9,Kimi no Suizou wo Tabetai,"[drama, romance]",1.0,0.929702,0.978911


In [50]:
df.loc[df["Name"] == "Horimiya"]

,anime_id,Name,English name,Other name,Score,Genres,Synopsis,Type,Episodes,Aired,...,Rating,Rank,Popularity,Favorites,Scored By,Members,Image URL,Score_missing,weighted_score,final_score
17199,42897,Horimiya,Horimiya,ホリミヤ,8.2,Romance,"On the surface, the thought of Kyouko Hori and...",TV,13.0,"Jan 10, 2021 to Apr 4, 2021",...,PG-13 - Teens 13 or older,350.0,0.995955,0.146696,745655.0,0.331885,https://cdn.myanimelist.net/images/anime/1695/...,0,0.865855,0.88537
